# Grading Records

Your mark sheet for the whole course. Every attempt at every test is logged in
`records.json`, and this notebook turns that ledger into a progress report:
per-module best scores, the weighted course mark, the band, and the next
module to attack.

- `records.json` - the ledger (plain JSON, safe to commit, yours to edit).
- `grade_records.py` - the command line tool that writes it and reports on it.
- `GRADE_REPORT.md` - the exported markdown report, ready for your portfolio.

The cells below are live: they read the ledger every time you run them.

## 1. Recording an attempt

**After you have worked through a test in Jupyter**, grade it and log it in one
step:

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py run "T03 - Methods and Functions Test.ipynb" --prose 7
```

The grader replays your saved notebook, counts the checks that pass, then asks
for the points for the hand-marked questions (`--prose`). Those are the
multiple-choice and conceptual questions: 1 point each.

**If you already know your numbers**, record them directly:

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py record T00 --checks 5/6 --prose 6 --notes "slicing still slips"
```

**Read it back**

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py list
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py summary
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py report
```

**Start a clean attempt for one module**

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py reset T03
```

Your best score per module is kept, so a retake can only improve your record.

## 2. Live report

Run the cell below to see the current state of your record.

In [ ]:
import os
import sys

HERE = os.path.dirname(os.path.abspath(".")) if False else os.getcwd()
if not os.path.exists(os.path.join(HERE, "grade_records.py")):
    HERE = r"C:/Users/andre/Learning/Udemy-Complete-Python-3-Bootcamp/20-Testing-Syllabus"
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import grade_records as gr

try:
    from IPython.display import Markdown, display

    def show_md(text):
        display(Markdown(text))
except (ImportError, NameError):
    def show_md(text):
        print(text)

**Course mark and band**

In [ ]:
summary = gr.summarize()

lines = [
    "| Measure | Value |",
    "| --- | --- |",
    "| Course mark | **%.1f / 100** (%s) |" % (summary["course_mark"] * 100, summary["band"]),
    "| Section tests | %.1f / 100  (85%% of the mark) |" % (summary["section_mark"] * 100),
    "| Final exam | %s  (15%% of the mark, pass %d/60) |" % (
        ("%.1f / 100" % (summary["final_mark"] * 100)) if summary["final_points"] is not None else "not attempted",
        gr.FINAL_PASS),
    "| Modules signed off | %d / %d (pass mark %d/20) |" % (
        len(summary["signed_off"]), len(summary["rows"]), gr.PASS_MARK),
    "| Attempts logged | %d |" % len(summary["attempts"]),
]
if summary["next_target"]:
    lines.append("| Next target | %s |" % summary["next_target"])

show_md("\n".join(lines))

**Per-module best scores**

In [ ]:
rows = ["| Test | Module | Best | Weight | Progress |", "| --- | --- | --- | --- | --- |"]
for row in summary["rows"]:
    test = row["test"]
    best = row["best"]
    mark = ("%d / %d" % (best, test["points_possible"])) if best is not None else "- / %d" % test["points_possible"]
    state = "signed off" if best is not None and best >= gr.PASS_MARK else (
        "attempted" if best is not None else "not attempted")
    rows.append("| %s | %s | %s | %d%% | %s %s |" % (
        test["id"], test["title"], mark, test["weight"], gr.format_bar(best, test["points_possible"]), state))

show_md("\n".join(rows))

**Attempt history**

Every attempt is kept, including the ones you redid. Nothing is deleted.

In [ ]:
if not summary["attempts"]:
    show_md("No attempts recorded yet. Run `grade_records.py run ...` after your first test.")
else:
    best = summary["best"]
    rows = ["| Date | Test | Attempt | Checks | Coding | Prose | Total | Status | Supersedes | Chain hash | Notes |",
            "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |"]
    for attempt in summary["attempts"]:
        status = "current best" if best.get(attempt["test"]) == attempt["points"] else "superseded"
        rows.append("| %s | %s | %d | %d/%d | %d | %d | **%d / %d** | %s | %s | `%s` | %s |" % (
            attempt["date"], attempt["test"], attempt.get("attempt_number", 1),
            attempt["checks_passed"], attempt["checks_total"],
            attempt["coding_points"], attempt["prose_points"], attempt["points"],
            attempt["points_possible"], status,
            ("#%d" % attempt["supersedes"]) if attempt.get("supersedes") else "-",
            attempt["hash"][:12], attempt["notes"] or "-"))
    show_md("\n".join(rows))

## 3. Redoing a test or a project

A mark you are unhappy with is a normal reason to try again. The ledger is
built for exactly that:

- **Nothing is deleted.** The ledger is append-only. A redo is a new entry
  that names the attempt it supersedes and carries its attempt number.
- **The best attempt counts.** A worse retake can never lower your mark.
- **A reset voids, it does not erase.** `grade_records.py reset T03` appends
  a `reset` event; the earlier attempts stay in the history, marked void.
  Use it when a test was broken, not to hide a bad score.
- **The improvement is the story.** Both attempts appear in `GRADE_REPORT.md`
  and in the table above. A reviewer sees you went from 12/20 to 19/20, which
  says more than a lone 19/20.

Worked example:

```
grade_records.py run "T03 - Methods and Functions Test.ipynb" --prose 5
    -> attempt 1, 14/20, chain hash a1b2c3...

# revise, then redo
grade_records.py run "T03 - Methods and Functions Test.ipynb" --prose 8
    -> attempt 2, 19/20, supersedes #1, chain hash f6e5d4...

grade_records.py list
    2026-11-13  T03  attempt 1                 4/6 checks  14/20 points  superseded  a1b2c3
    2026-11-20  T03  attempt 2 (supersedes #1) 6/6 checks  19/20 points  best        f6e5d4
```

### The same applies to the projects

The portfolio in `S3 - Project Portfolio.ipynb` has the same rule: a project
you reworked keeps its earlier version in git history. Commit the revision
rather than rewriting the old commit - the diff is the evidence that you
improved it, and `INTEGRITY.md` explains how the seal ties the two together.

**Sign-off checklist**

A module is signed off at 14 / 20. Tick them here, or in the readiness
checklist at the end of `S0 - Testing Syllabus.ipynb`.

In [ ]:
rows = []
for row in summary["rows"]:
    test = row["test"]
    best = row["best"]
    done = best is not None and best >= gr.PASS_MARK
    rows.append("- [%s] **%s** %s - %d/%d%s" % (
        "x" if done else " ", test["id"], test["title"], best if best is not None else 0,
        test["points_possible"], " (signed off)" if done else ""))
rows.append("")
if summary["final_points"] is not None:
    final_done = summary["final_points"] >= gr.FINAL_PASS
    rows.append("- [%s] **FINAL** Final Comprehensive Exam - %d/60%s" % (
        "x" if final_done else " ", summary["final_points"], " (signed off)" if final_done else ""))
else:
    rows.append("- [ ] **FINAL** Final Comprehensive Exam - not attempted")

show_md("\n".join(rows))

**Project work**

Projects are graded with the shared rubric in `rubrics.json` - six criteria,
scored 0-4 with weights, the same for every project in the course.

In [ ]:
if not summary["projects"]:
    show_md("No projects graded yet.\n\n"
            "Score one with:\n\n"
            "```\n"
            "grade_records.py rubric P05\n"
            "grade_records.py record-project P05 --scores 3,4,2,3,4,3\n"
            "```")
else:
    rows = ["| Project | Title | Mark | Level | Rubric scores | Date |", "| --- | --- | --- | --- | --- | --- |"]
    for project_id, entry in sorted(summary["projects"].items()):
        scores = ", ".join("%s %d/%d" % (item["name"], item["score"], item["max"])
                           for item in entry.get("scores", []))
        rows.append("| %s | %s | **%s%%** | %s | %s | %s |" % (
            project_id, entry["title"], entry["percent"], entry["level"], scores, entry.get("date", "-")))
    rows.append("")
    rows.append("Portfolio average: **%s%%** across %d graded project(s)." % (
        summary["portfolio_mark"], len(summary["projects"])))
    show_md("\n".join(rows))

**Where the work is weakest**

The lowest average across your graded projects is where a reviewer would look
first - this is your next target, computed from your own scores.

In [ ]:
if not summary["weakest_criteria"]:
    show_md("Grade a project to see this breakdown.")
else:
    names = gr.criterion_names()
    rows = ["| Criterion | Average score (0-4) | Projects |", "| --- | --- | --- |"]
    for item in summary["weakest_criteria"]:
        rows.append("| %s | %.2f | %d |" % (
            names.get(item["criterion"], item["criterion"].replace("_", " ").capitalize()),
            item["average"], item["projects"]))
    weakest = summary["weakest_criteria"][0]
    rows.append("")
    rows.append("**Work on this next:** %s (%.2f / 4 across %d project(s))." % (
        names.get(weakest["criterion"], weakest["criterion"].replace("_", " ").capitalize()),
        weakest["average"], weakest["projects"]))
    show_md("\n".join(rows))

**Progress journal**

A dated log of what you actually did. It is a chained ledger entry, so it
cannot be quietly rewritten either.

```
grade_records.py log "rebuilt the OOP model, still stuck on super()"
```

In [ ]:
if not summary["journal"]:
    show_md("No journal entries yet. Add one with `grade_records.py log \"what you did\"`.")
else:
    rows = ["| When | Note |", "| --- | --- |"]
    for entry in summary["journal"]:
        rows.append("| %s | %s |" % (entry.get("date", "-"), entry.get("text", "").replace("|", "/")))
    show_md("\n".join(rows))

## 4. How the course mark is calculated

Each module test is worth 20 points and carries a weight from the blueprint in
`S0 - Testing Syllabus.ipynb`. Your section mark is the weighted average of
your **best** attempt at each module:

```
section mark = sum(best_points * weight) / sum(20 * weight)      -> 85% of the course mark
final exam   = best_points / 60                                  -> 15% of the course mark
course mark  = 0.85 * section mark + 0.15 * final exam
```

Bands: Distinction 85+, Merit 70+, Pass 55+, below pass under 55.

**Why the best attempt counts.** Retakes are how learning works here. The
ledger keeps every attempt so you can see the trend, but only your best score
per module counts towards the mark.

## 5. The ledger file

`records.json` holds one entry per attempt:

```json
{
  "entries": [
    {
      "seq": 1,
      "prev": "0000000000000000000000000000000000000000000000000000000000000000",
      "event": "attempt",
      "attempt_number": 1,
      "date": "2026-11-13",
      "test": "T03",
      "title": "Methods and Functions",
      "checks_passed": 6,
      "checks_total": 6,
      "coding_points": 12,
      "prose_points": 7,
      "points": 19,
      "points_possible": 20,
      "notes": "kwargs finally clicked",
      "notebook": "T03 - Methods and Functions Test.ipynb",
      "notebook_sha256": "9f2c...e71a",
      "notebook_git_blob": "1a2b3c4d",
      "hash": "7ca83b6b5f5bcb76..."
    }
  ]
}
```

It is plain JSON: read it, edit it, or keep it in a spreadsheet tool. The
notebook and the CLI both read the same file, so they always agree.

Two fields make it tamper-evident. `prev` is the hash of the previous entry
and `hash` is the hash of this entry's own content, so the entries form a
chain: editing any past entry breaks every link after it. `notebook_sha256`
and `notebook_git_blob` pin the exact file that was graded, so a later edit
to your answer cells is visible. `INTEGRITY.md` explains the whole scheme and
what it can and cannot prove.

## 6. Using the record as portfolio evidence

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py report
```

writes `GRADE_REPORT.md`: the course mark, the per-module table, the attempt
history and the signed-off checklist. Commit it to your portfolio repository
under `docs/` and it becomes a dated record of your progress through the
course - the same pattern described in `S3 - Project Portfolio.ipynb`.